# CareLoop Clinics – Part 4: Generative AI Follow-up Analysis & Recommendations
**Tasks 4.1 – 4.4: Gemini Structured Extraction, Pydantic Validation, Prompt Testing & Synthesis**

This notebook uses Google's free Gemini API (`google-genai`) in JSON mode with a 4-field schema and Pydantic validation to analyse the first 10 follow-up notes, tests prompt reliability, and integrates all four project parts.

## Task 4.1 – One Validated Call (5 Marks)
Install/import `google-genai` and `pydantic`, define the 4-field `SCHEMA` (no enum on `root_cause`), request the API key at runtime via `input()`, build the `CRAFT` prompt, implement `analyse_note(note_text)`, and test on `N001`.

In [ ]:
try:
    from google import genai
except ImportError:
    import importlib, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-genai", "pydantic"])
    sys.modules.pop("google", None)
    importlib.invalidate_caches()
    from google import genai

import json
import os
import time
import pandas as pd
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

def resolve_input_file(candidates):
    search_roots = [
        ".", "/content", "..",
        "data/raw", "/content/data/raw", "../data/raw",
        "data/processed", "/content/data/processed", "../data/processed",
        "outputs", "/content/outputs", "../outputs",
        "CapstoneProjectMasai", "CapstoneProjectMasai/data/processed",
    ]
    for root in search_roots:
        for name in candidates:
            path = os.path.join(root, name)
            if os.path.isfile(path):
                return path
    for root_dir in (".", "/content"):
        if os.path.isdir(root_dir):
            for r, _, files in os.walk(root_dir):
                for name in candidates:
                    if name in files:
                        return os.path.join(r, name)
    raise FileNotFoundError(
        f"Could not find any of {candidates} under {os.getcwd()} or /content. "
        f"Files in {os.getcwd()}: {os.listdir('.')}. "
        f"Please upload {candidates[0]} into Colab's /content/ folder."
    )

PROJECT_ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

NOTES_PATH = resolve_input_file(["followup_notes.csv", "followupnotes.csv"])
APPTS_CLEAN_PATH = resolve_input_file(["appointments_clean.csv"])
print("Follow-up notes path   :", NOTES_PATH)
print("Clean appointments path:", APPTS_CLEAN_PATH)

In [ ]:
# STEP B - JSON response schema (root_cause is a plain string, no enum)
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
    ],
}

ALLOWED_CATEGORIES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

# STEP C - Runtime API key (never hardcode, print, or save)
API_KEY = input("Paste your Gemini API key: ").strip()
client = genai.Client(api_key=API_KEY)

# STEP D - Gemini Flash model
MODEL_NAME = "gemini-3.8-flash"

# STEP E - System instruction
SYSTEM_TEXT = (
    "You are analysing outpatient appointment follow-up notes for CareLoop Clinics. "
    "Identify the most likely reason for the missed appointment and select exactly one of the "
    "six allowed categories: forgot_or_no_reminder, distance_or_transport, cost_or_fee, "
    "illness_or_emergency, work_or_schedule, or other. "
    "Do not invent patient information. If the reason is unclear, choose 'other' instead of guessing. "
    "Return only the requested structured JSON."
)

api_call_count = 0

def build_craft_prompt(note_text):
    """Build the CRAFT prompt (Context, Role, Action, Format, Tone)."""
    return (
        "Context: CareLoop Clinics operates outpatient clinics and collects call-centre notes "
        "after missed appointments.\n"
        "Role: Act as a healthcare appointment follow-up analyst.\n"
        "Action: Read the supplied note and classify the most likely reason for the missed "
        "appointment into one of these exact six categories: "
        "forgot_or_no_reminder, distance_or_transport, cost_or_fee, "
        "illness_or_emergency, work_or_schedule, other. "
        "If the reason is unclear or ambiguous, choose 'other' instead of guessing. "
        "Suggest a practical follow-up action.\n"
        "Format: Return JSON with exactly four fields: root_cause, confidence (0 to 1), "
        "summary (maximum 25 words), and recommended_action.\n"
        "Tone: Be concise, factual and neutral. Do not invent details.\n\n"
        f"Patient follow-up note: {note_text}"
    )

def call_gemini_json(prompt_text):
    """Call Gemini Flash in JSON mode (thinking_config removed because gemini-3.8-flash rejects minimal)."""
    global api_call_count
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
    )
    for attempt in range(4):
        try:
            api_call_count += 1
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
                config=config,
            )
        except Exception as exc:
            if "503" in str(exc) and attempt < 3:
                wait_s = 10 * (attempt + 1)
                print(f"503 UNAVAILABLE (high demand), retrying in {wait_s}s...")
                time.sleep(wait_s)
                continue
            raise

def analyse_note(note_text):
    """Send one patient note to Gemini Flash using CRAFT and return parsed JSON."""
    response = call_gemini_json(build_craft_prompt(note_text))
    return json.loads(response.text)

In [ ]:
# STEP H - Test N001
notes_df = pd.read_csv(NOTES_PATH)
n001_row = notes_df.iloc[0]
print("Testing note:", n001_row["note_id"], "|", n001_row["note_text"])

n001_result = analyse_note(n001_row["note_text"])
print("\nParsed dictionary:")
print(json.dumps(n001_result, indent=2))
print("Field count:", len(n001_result))
print("N001 root_cause == 'forgot_or_no_reminder':", n001_result.get("root_cause") == "forgot_or_no_reminder")
print("API calls so far:", api_call_count)

## Task 4.2 – Validate and Process the 10 Notes (7 Marks)
Define `FollowupInsight` in Pydantic, validate `root_cause` separately against `ALLOWED_CATEGORIES`, skip blank notes (`N004`) without calling Gemini, pause 13 seconds between API calls, preserve invalid responses as-is, and export `data/processed/notes_analysed.csv`.

In [ ]:
# STEP A - Define Pydantic model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# STEP C - Process only the first 10 rows
top10_notes = notes_df.head(10).copy()
results = []

for idx, row in top10_notes.iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]

    # STEP D - Handle blank notes without calling Gemini
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    # STEP E - Call Gemini and validate response without repairing invalid outputs
    raw_text = ""
    try:
        response = call_gemini_json(build_craft_prompt(note_text))
        raw_text = response.text or ""
        data = json.loads(raw_text)
    except Exception as exc:
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "",
            "confidence": None,
            "summary": "",
            "recommended_action": "",
            "validation_status": "invalid",
            "raw_response": raw_text if raw_text else str(exc),
        })
        time.sleep(13)
        continue

    try:
        validated = FollowupInsight(**data)
        if validated.root_cause not in ALLOWED_CATEGORIES:
            raise ValueError(f"Disallowed root_cause: {validated.root_cause}")
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": validated.root_cause,
            "confidence": validated.confidence,
            "summary": validated.summary,
            "recommended_action": validated.recommended_action,
            "validation_status": "valid",
            "raw_response": raw_text,
        })
    except (ValidationError, ValueError):
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", None),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_text,
        })

    # STEP F - Rate limiting after real API calls only
    time.sleep(13)

# STEP G & H - Build exact 8-column DataFrame and export
OUTPUT_COLS = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response",
]
notes_analysed = pd.DataFrame(results)[OUTPUT_COLS]
notes_analysed_path = os.path.join(PROCESSED_DIR, "notes_analysed.csv")
notes_analysed.to_csv(notes_analysed_path, index=False)

print("Saved:", notes_analysed_path, "| Shape:", notes_analysed.shape)
print(notes_analysed[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
print("API calls after Task 4.2:", api_call_count)

## Task 4.3 – Break It on Purpose (5 Marks)
Test Pydantic `ValidationError` on invalid confidence (`1.7`) and missing `recommended_action`, compare simple prompt `V1` vs. CRAFT prompt `V2` on `N007` (Hinglish) and `N009` (ambiguous) with identical model/schema/temperature settings, and verify total API calls remain `< 25`.

In [ ]:
# STEP A - Deliberately invalid Pydantic tests
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send SMS reminder.",
}
try:
    FollowupInsight(**bad_reply_1)
except ValidationError as e:
    print("TEST 1 (confidence=1.7) caught ValidationError:")
    print(e)

bad_reply_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.85,
    "summary": "Patient skipped due to consultation fee.",
}
try:
    FollowupInsight(**bad_reply_2)
except ValidationError as e:
    print("\nTEST 2 (missing recommended_action) caught ValidationError:")
    print(e)

# STEP B & C - Compare V1 ("Classify this note") vs V2 (CRAFT) on N007 and N009
n007_text = notes_df.loc[notes_df["note_id"] == "N007", "note_text"].iloc[0]
n009_text = notes_df.loc[notes_df["note_id"] == "N009", "note_text"].iloc[0]

def run_v1_prompt(note_text):
    prompt = f"Classify this note\n\nNote: {note_text}"
    return json.loads(call_gemini_json(prompt).text)

prompt_comp_rows = []
for nid, txt in [("N007", n007_text), ("N009", n009_text)]:
    v1_res = run_v1_prompt(txt)
    time.sleep(13)
    v2_res = analyse_note(txt)
    time.sleep(13)
    for ver, res in [("V1", v1_res), ("V2", v2_res)]:
        rc = res.get("root_cause", "")
        prompt_comp_rows.append({
            "Note ID": nid,
            "Prompt Version": ver,
            "Root Cause": rc,
            "Allowed Category?": "Yes" if rc in ALLOWED_CATEGORIES else "No",
        })

prompt_comp_df = pd.DataFrame(prompt_comp_rows)
print("\nPrompt Comparison (V1 vs V2):")
print(prompt_comp_df.to_string(index=False))
print("\nTotal API calls across Tasks 4.1-4.3:", api_call_count)
assert api_call_count < 25, f"Exceeded 25-call limit: {api_call_count}" 

**Prompt Comparison Explanation (Task 4.3):** The CRAFT prompt (`V2`) is far more reliable for production deployment than the minimal `V1` prompt because it explicitly defines the clinic context, analyst role, the exact six allowed `root_cause` labels, and the instruction to choose `other` rather than guessing—reducing off-schema or invented categories.

## Task 4.4 – Merge Structured Reasons with Appointment Data (5 Marks)
Merge `notes_analysed.csv` with `appointments_clean.csv` on `appointment_id` (10 rows) and cross-tabulate `root_cause` against `reminder_sent`.

In [ ]:
appts_clean = pd.read_csv(APPTS_CLEAN_PATH)
merged_df = notes_analysed.merge(appts_clean, on="appointment_id", how="inner")
print("Merged shape (expected 10 rows):", merged_df.shape)

ct = pd.crosstab(merged_df["root_cause"], merged_df["reminder_sent"])
print("\nCross-tabulation of root_cause vs reminder_sent:")
print(ct)